# CIE 2 — Activity 1 & 2
## Regression and Classification Algorithms
**Student:** Himanshu Badole  
**Course:** Regression and Classification Algorithms  
**Environment:** Google Colab / Jupyter Notebook

This notebook covers:
- **Activity 1:** Multiple Linear Regression for a continuous target
- **Activity 2:** Logistic Regression for binary classification

> Run all cells from top to bottom. The outputs and metrics are generated by the code; do not copy or claim results until you execute the notebook.

# Activity 1 — Multiple Linear Regression
## 1. Problem statement and dataset
**Role:** Data analyst studying factors associated with diabetes disease progression.

We use the built-in Scikit-learn Diabetes dataset. Each row represents a patient record with numerical baseline measurements. The target is a quantitative measure of disease progression one year later (continuous).

**Features used for the full model:** BMI, blood pressure, a blood serum measurement (`s5`), and (`s6`).  
**Target (`y`):** disease progression measure.

This dataset is for statistical learning practice only—not diagnosis or clinical decisions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
data = load_diabetes(as_frame=True)
df = data.frame.copy()

print("Dataset:", data.DESCR.splitlines()[0])
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
display(df.head())
print("\\nData types:")
print(df.dtypes)
print("\\nSummary statistics:")
display(df.describe().T)
print("\\nMissing values:")
print(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

## 2. Data cleaning and feature selection
This dataset is already numeric and is provided in a cleaned form by Scikit-learn. We check missing values and duplicates. We do not use an identifier column because there is none.

We select four numeric inputs for the multiple-feature model: `bmi`, `bp`, `s5`, and `s6`. They represent baseline numerical measurements. The feature selection is for demonstration and is not a claim of medical causation.

In [ ]:
features = ["bmi", "bp", "s5", "s6"]
target = "target"

print("Selected features:", features)
print("Target:", target)
print("Missing in selected data:", df[features + [target]].isna().sum().sum())

## 3. Visual exploration
Scatter plots show how each selected input relates to the continuous target. Correlation measures linear association, not causation.

In [ ]:
for col in features:
    plt.figure(figsize=(6,4))
    plt.scatter(df[col], df[target], alpha=0.5)
    plt.title(f"{col} vs disease progression target")
    plt.xlabel(col)
    plt.ylabel("Target (continuous progression measure)")
    plt.grid(alpha=0.2)
    plt.show()

corr = df[features + [target]].corr()
plt.figure(figsize=(7,5))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap — Activity 1")
plt.show()

print("Feature-target correlations:")
display(corr[target].drop(target).sort_values(key=lambda s: s.abs(), ascending=False))

## 4. Manual multiple regression calculation (small sample)
For learning the normal-equation method, we use **5 observations and two features** (`bmi` and `bp`) from the dataset.

The model is:
\[
y = b_0 + b_1 x_1 + b_2 x_2
\]

In matrix form:
\[
\mathbf{b}=(X^TX)^{-1}X^Ty
\]

The next cell prints the sample, design matrix \(X\), target vector \(y\), \(X^TX\), \(X^Ty\), and the calculated coefficients. These are the values to copy into your manual worksheet, showing the matrix steps.

In [ ]:
small = df[["bmi", "bp", "target"]].head(5).copy()
X_small = np.column_stack([np.ones(len(small)), small["bmi"].values, small["bp"].values])
y_small = small["target"].values

XtX = X_small.T @ X_small
Xty = X_small.T @ y_small
b_manual = np.linalg.solve(XtX, Xty)

display(small)
print("Design matrix X (columns: 1, BMI, BP):\\n", X_small)
print("\\nTarget vector y:\\n", y_small)
print("\\nX^T X:\\n", XtX)
print("\\nX^T y:\\n", Xty)
print("\\nManual coefficients [b0, b1, b2]:\\n", b_manual)
print(f"Manual fitted equation: y = {b_manual[0]:.4f} + ({b_manual[1]:.4f})*BMI + ({b_manual[2]:.4f})*BP")

# Predict first sample and calculate residual (actual - predicted)
pred_first = X_small[0] @ b_manual
res_first = y_small[0] - pred_first
print(f"First observation: actual={y_small[0]:.4f}, predicted={pred_first:.4f}, residual={res_first:.4f}")

### Verify manual calculation with Scikit-learn
We fit a library Linear Regression model on the exact same 5 records and same two features. Its coefficients should match the normal-equation result up to numerical rounding.

In [ ]:
manual_verify = LinearRegression()
manual_verify.fit(small[["bmi", "bp"]], small["target"])
print("Library intercept:", manual_verify.intercept_)
print("Library coefficients:", manual_verify.coef_)
print("Manual intercept:", b_manual[0])
print("Manual coefficients:", b_manual[1:])

## 5. Train/test split and models
We use an 80:20 train/test split. The test set is held aside for final evaluation. The single-feature model uses the feature with the strongest absolute correlation with the target in the training data. The multiple model uses the four selected inputs.

In [ ]:
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

train_corr = X_train.join(y_train).corr()[target].drop(target)
best_feature = train_corr.abs().idxmax()
print("Strongest training feature by absolute correlation:", best_feature)

single_model = LinearRegression().fit(X_train[[best_feature]], y_train)
multi_model = LinearRegression().fit(X_train, y_train)

print("\\nSingle-feature model:")
print("Intercept:", single_model.intercept_)
print("Coefficient:", single_model.coef_[0])
print("Multiple-feature model:")
print("Intercept:", multi_model.intercept_)
print(pd.Series(multi_model.coef_, index=features, name="Coefficient"))

## 6. Interpret the multiple regression equation
The equation is assembled below from the fitted model. A coefficient means the predicted change in the target for a one-unit increase in that feature, **holding the other included features constant**. Coefficients should not be interpreted as causal effects. The intercept is the predicted target when all input values are zero; this may not be a realistic record.

In [ ]:
equation = f"Target = {multi_model.intercept_:.4f}"
for name, coef in zip(features, multi_model.coef_):
    equation += f" + ({coef:.4f})*{name}"
print(equation)

print("\\nCoefficient interpretation (holding other features constant):")
for name, coef in zip(features, multi_model.coef_):
    direction = "increases" if coef >= 0 else "decreases"
    print(f"A one-unit increase in {name} is associated with a {direction} of {abs(coef):.4f} in the predicted target.")

## 7. Test predictions and evaluation
We compare the single-feature and multiple-feature models using test-set MAE, MSE, RMSE, R², and adjusted R². Lower MAE/RMSE is better; higher R² is generally better. Test R² can be negative if the model performs worse than predicting the test-set mean.

In [ ]:
def regression_metrics(y_true, y_pred, n_features):
    n = len(y_true)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    adj_r2 = 1 - (1-r2)*(n-1)/(n-n_features-1) if n > n_features + 1 else np.nan
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2, "Adjusted R2": adj_r2}

pred_single = single_model.predict(X_test[[best_feature]])
pred_multi = multi_model.predict(X_test)

comparison = pd.DataFrame({
    "Single-feature": regression_metrics(y_test, pred_single, 1),
    "Multiple-feature": regression_metrics(y_test, pred_multi, len(features))
}).T
display(comparison)

prediction_table = pd.DataFrame({"Actual": y_test, "Single-feature predicted": pred_single,
                                "Multiple-feature predicted": pred_multi})
display(prediction_table.head(10))

## 8. Residual analysis and multicollinearity
Residual = actual − predicted. Residual plots help identify patterns that linear regression may not capture. VIF is a diagnostic for multicollinearity; large values suggest that features overlap strongly in the information they provide.

In [ ]:
residuals = y_test - pred_multi

plt.figure(figsize=(6,4))
plt.scatter(pred_multi, residuals, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.title("Residuals vs Predicted — Multiple Regression")
plt.xlabel("Predicted target")
plt.ylabel("Residual (actual − predicted)")
plt.show()

plt.figure(figsize=(6,4))
plt.hist(residuals, bins=20, edgecolor="black")
plt.title("Residual Histogram")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(6,4))
plt.scatter(y_test, pred_multi, alpha=0.6)
lims = [min(y_test.min(), pred_multi.min()), max(y_test.max(), pred_multi.max())]
plt.plot(lims, lims, "--")
plt.title("Actual vs Predicted — Multiple Regression")
plt.xlabel("Actual target")
plt.ylabel("Predicted target")
plt.show()

# VIF computed using statsmodels if available
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    X_vif = X_train.reset_index(drop=True)
    vif_table = pd.DataFrame({
        "Feature": X_vif.columns,
        "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    })
    display(vif_table)
except ImportError:
    print("Install statsmodels to calculate VIF: !pip install statsmodels")

## 9. New prediction and conclusion
Choose a new example within the observed range of the training data. The notebook uses the median feature values as a reasonable illustrative input. This is an estimate, not a guaranteed outcome.

In [ ]:
new_record = X_train.median().to_frame().T
new_prediction = multi_model.predict(new_record)[0]
print("New example (median feature values):")
display(new_record)
print(f"Predicted target: {new_prediction:.3f}")

print("\\nConclusion prompts:")
print("- Compare the multiple model against the single-feature model using test MAE/RMSE/R2.")
print("- Mention whether the extra features improved test performance enough to justify complexity.")
print("- Discuss residual patterns and multicollinearity/VIF.")
print("- State that linearity, independence, constant variance, normal residuals and no severe multicollinearity are assumptions to assess.")
print("- This educational model is not suitable for clinical decisions.")